In [ ]:
from pathlib import Path
import json
from typing import List

import numpy as np
import pandas as pd
import h5py

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
repo_dir = Path("../../..")
results_dir = repo_dir / 'extract_n_eval' / "results" / "proj_feats"
save_dir = repo_dir / "visualization" / "paper" / "supp" / "figures"

assert results_dir.exists(), f"data directory {results_dir} does not exist!"

In [ ]:
MODEL_NAME = "deit_small_imagenet_full_seed-0"
# MODEL_NAME = "deitflex-l-1_imagenet_full_seed-0"

X_KEY = "layer_position_normalized"
Y_KEYS = [
    "pearsonr_nc",
    "cv_score",
]

FN_FILTER_LAYERS = lambda x: (
    False
    # or x.endswith("norm1") 
    or x.endswith("norm2") 
    # or x.endswith("mlp") 
    # or x.endswith("attn")
)
FN_FILTER_LAYERS = lambda x: True  # No filter

In [ ]:
valid_filenames = [
    "bs_fz.json",
    "bs_mh.json",
    "tvsd.json",
    "things_fmri.json",
    "things_eeg1.json",
    "things_eeg2.json",
    "things_meg.json",
    "nsd_func1pt8mm_individualROIs.json",
    "nsd_fsaverage_individualROIs.json",
    "nsd_nativesurface_individualROIs.json",
]

found_results_files = [file for file in results_dir.glob("**/*.json") if file.name in valid_filenames]

all_results = []

for file in tqdm(found_results_files):
    try:
        with open(file, 'r') as f:
            result = json.load(f)
    except Exception as e:
        print(f"Error loading {file}: {e}")
        continue
    all_results.extend(result["results"])
    
df_all_results = pd.DataFrame(all_results)
df_all_results

In [ ]:
METRIC_NAMES_MAP = {
    "pearsonr_nc": "Pearson's r (NC)",
    "cv_score": "Cross-validated score (NC)",
    "cka_c_test": "CKA-C (test)",
    "rsa_c_test": "RSA-C (test)",
    "cka_ve_test": "CKA-VE (test)",
    "rsa_ve_test": "RSA-VE (test)",
    "cka_c_train": "CKA-C (train)",
    "rsa_c_train": "RSA-C (train)",
    "cka_ve_train": "CKA-VE (train)",
    "rsa_ve_train": "RSA-VE (train)",
}

BENCHMARK_NAMES_MAP = {
    "nsd_fsaverage_individualROIs": "NSD fMRI - fsaverage",
    "nsd_func1pt8mm_individualROIs": "NSD fMRI - Native Volume",
    "nsd_nativesurface_individualROIs": "NSD fMRI - Native Surface",
    "things_fmri": "THINGS fMRI",
    "things_eeg1": "THINGS EEG-1",
    "things_eeg2": "THINGS EEG-2",
    "things_meg": "THINGS MEG",
    "tvsd": "TVSD",
    "bs_fz": "Brain-Score FZ",
    "bs_mh": "Brain-Score MH",
}

BENCHMARK_NAMES_MAP = {
    "nsd_fsaverage_individualROIs": "NSD fMRI - fsaverage",
    "nsd_func1pt8mm_individualROIs": "NSD-fMRI",
    "nsd_nativesurface_individualROIs": "NSD fMRI - Native Surface",
    "things_fmri": "T-fMRI",
    "things_eeg1": "T-EEG1",
    "things_eeg2": "T-EEG2",
    "things_meg": "T-MEG",
    "tvsd": "TVSD-EP",
    "bs_fz": "FZ-EP",
    "bs_mh": "MH-EP",
}

In [ ]:
def get_metric_colors(metrics: List[str]) -> dict:
    base_colors = sns.color_palette("tab10", n_colors=len(metrics))
    metric_colors = {metric: base_colors[i] for i, metric in enumerate(metrics)}
    return metric_colors

In [ ]:
for dataset in df_all_results['benchmark_name'].unique():
    available_rois = df_all_results[df_all_results['benchmark_name'] == dataset]['roi'].unique()
    print(f"{dataset}: {available_rois}")

In [ ]:
def save_figs(fig, save_dir, base_filename, dpi=300, formats=("png", "pdf", "svg")):
    for fmt in formats:
    
        save_fmt_dir = Path(save_dir) / fmt
        if not save_fmt_dir.exists():
            save_fmt_dir.mkdir(parents=True, exist_ok=False)
        
        file_path = save_fmt_dir / f"{base_filename}.{fmt}"
        
        fig.savefig(file_path, dpi=dpi, bbox_inches='tight')

        print(f"Figure saved to {file_path}")

In [ ]:
# ----------------------------
# 1) Super-ROI mapping
# ----------------------------
EARLY_ROIS = {
    "V1", "V2", "V1v", "V1d", "V2v", "V2d",
    "early",                 # NSD composite ROI
    # "occipital",             # EEG/MEG coarse ROI
}

MID_ROIS = {
    "V3", "V3v", "V3d", "V3a", "V3b",
    "V4", "hV4",
    "midventral", "midlateral", "midparietal",  # NSD composites
    # "occipital_parietal", "parietal",           # EEG/MEG coarse ROIs
}

def roi_to_superroi(benchmark_name: str, roi: str) -> str:
    """
    Map dataset-specific ROI names into {Early, Mid, High}.
    Everything not explicitly Early/Mid becomes High.
    """
    if roi in EARLY_ROIS:
        return "Early"
    if roi in MID_ROIS:
        return "Mid"
    return "High"


In [ ]:
from typing import Optional, List, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def compute_layerwise_rank_table(
    df_all: pd.DataFrame,
    dataset: str,
    metric: str = "cv_score",
    model_ids: Optional[List[str]] = None,
    layer_filter_fn=None,
    superroi: Optional[str] = None,  # "Early"|"Mid"|"High"|None
) -> pd.DataFrame:
    """
    Returns one row per (model_id, layer_position) for a given dataset:
      - depth (layer_position_normalized)
      - score (mean over subjects+ROIs)
      - rank_norm (layer rank by score within model, best=0 -> 0.0)
    If superroi is provided, first map ROI->superroi and aggregate within that superroi.
    """
    req = {"benchmark_name", "model_id", "layer_name", "layer_position", "roi", metric}
    missing = req - set(df_all.columns)
    assert not missing, f"Missing columns: {missing}"

    df = df_all[df_all["benchmark_name"] == dataset].copy()
    if model_ids is not None:
        df = df[df["model_id"].isin(model_ids)].copy()
    if layer_filter_fn is not None:
        df = df[df["layer_name"].apply(layer_filter_fn)].copy()
    if len(df) == 0:
        return pd.DataFrame(columns=["model_id","layer_position","depth","score","rank_norm"])

    # normalize depth per model
    max_layer = df.groupby("model_id")["layer_position"].transform("max")
    df["depth"] = df["layer_position"] / max_layer

    # optional: map ROI -> superroi then filter
    if superroi is not None:
        df["superroi"] = [roi_to_superroi(bn, r) for bn, r in zip(df["benchmark_name"].values, df["roi"].values)]
        df = df[df["superroi"] == superroi].copy()
        if len(df) == 0:
            return pd.DataFrame(columns=["model_id","layer_position","depth","score","rank_norm"])

    # layerwise score (mean over subjects + ROIs)
    layerwise = (
        df.groupby(["model_id", "layer_position", "depth"])[metric]
          .mean()
          .reset_index()
          .rename(columns={metric: "score"})
    )

    # rank layers within each model by score (best=rank 0)
    layerwise["rank"] = layerwise.groupby("model_id")["score"].rank(ascending=False, method="average")

    # normalize to [0,1] per model
    n_layers = layerwise.groupby("model_id")["layer_position"].transform("nunique")
    denom = (n_layers - 1).replace(0, np.nan)
    layerwise["rank_norm"] = (layerwise["rank"] - 1) / denom  # best -> 0
    layerwise["rank_norm"] = 1 - layerwise["rank_norm"] # best -> 1

    return layerwise[["model_id", "layer_position", "depth", "score", "rank_norm"]]

In [ ]:
import scipy.stats as stats
def plot_panel_B_layer_order_scatter(
    df_all: pd.DataFrame,
    dataset_a: str,
    dataset_b: str,
    metric: str = "cv_score",
    mode: str = "rank",               # "rank" or "score"
    superrois: Tuple[str, ...] = ("Early", "Mid", "High"),
    model_ids: Optional[List[str]] = None,
    hue: Optional[str] = "depth",
    layer_filter_fn=None,
    ncols: int = 3,
    figsize: Tuple[int, int] = (18, 5),
    alpha: float = 0.15,
    s: float = 8,
):
    """
    Creates 1 row of subplots, one per superROI (or fewer).
    Each subplot scatters per-layer quantities between two datasets, over all models+layers.

    mode="rank": x=rank_norm(A), y=rank_norm(B)  (normalized layer ordering)
    mode="score": x=score(A),     y=score(B)      (raw layerwise scores)
    """
    assert mode in {"rank", "score"}

    n = len(superrois)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize, squeeze=False, sharex=False, sharey=False)
    axes = axes.flatten()

    for i, sr in enumerate(superrois):
        ax = axes[i]

        tab_a = compute_layerwise_rank_table(
            df_all=df_all, dataset=dataset_a, metric=metric, model_ids=model_ids,
            layer_filter_fn=layer_filter_fn, superroi=sr
        )
        tab_b = compute_layerwise_rank_table(
            df_all=df_all, dataset=dataset_b, metric=metric, model_ids=model_ids,
            layer_filter_fn=layer_filter_fn, superroi=sr
        )

        if len(tab_a) == 0 or len(tab_b) == 0:
            ax.text(0.5, 0.5, f"{sr}\n(no data)", ha="center", va="center")
            ax.set_axis_off()
            continue

        # merge on model_id + layer_position (same model layers across datasets)
        merged = tab_a.merge(
            tab_b,
            on=["model_id", "layer_position"],
            suffixes=("_a", "_b"),
            how="inner",
        )

        if len(merged) == 0:
            ax.text(0.5, 0.5, f"{sr}\n(no overlap)", ha="center", va="center")
            ax.set_axis_off()
            continue

        if mode == "rank":
            x = merged["rank_norm_a"].to_numpy()
            y = merged["rank_norm_b"].to_numpy()
            # ax.set_xlabel(f"{BENCHMARK_NAMES_MAP.get(dataset_a, dataset_a)}\n(rank-norm, best→0)")
            # ax.set_ylabel(f"{BENCHMARK_NAMES_MAP.get(dataset_b, dataset_b)}\n(rank-norm, best→0)")
            ax.set_xlabel(BENCHMARK_NAMES_MAP.get(dataset_a, dataset_a), fontsize=12, fontweight="bold")
            ax.set_ylabel(BENCHMARK_NAMES_MAP.get(dataset_b, dataset_b), fontsize=12, fontweight="bold")
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            # ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
        else:
            x = merged["score_a"].to_numpy()
            y = merged["score_b"].to_numpy()
            ax.set_xlabel(BENCHMARK_NAMES_MAP.get(dataset_a, dataset_a), fontsize=12, fontweight="bold")
            ax.set_ylabel(BENCHMARK_NAMES_MAP.get(dataset_b, dataset_b), fontsize=12, fontweight="bold")
            # ax.set_xlim(0, 1)
            # ax.set_ylim(0, 1)
            # no fixed limits; let it autoscale

        # correlation summary
        # print(np.corrcoef(x, y))
        print(stats.pearsonr(x, y))
        r = np.corrcoef(x, y)[0, 1] if (np.std(x) > 0 and np.std(y) > 0) else np.nan

        # Change color from early to deep layers
        if hue == "depth":
            depths = (merged["depth_a"] + merged["depth_b"]) / 2
            cmap = plt.get_cmap("viridis")
            colors = cmap(depths)
            ax.scatter(x, y, s=s, alpha=alpha, color=colors)
            sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=1))
            cbar = fig.colorbar(sm, ax=ax)
            cbar.set_label("Normalized layer depth", fontsize=12, fontweight="bold")
        else:
            ax.scatter(x, y, s=s, alpha=alpha)

        ax.set_title(f"{sr} (r={r:.2f}, n={len(merged)})", fontsize=16, fontweight="bold")
        ax.grid(True, linestyle="--", alpha=0.25)
        ax.spines[['top', 'right']].set_visible(False)

    # hide unused axes
    for j in range(len(superrois), len(axes)):
        axes[j].set_axis_off()

    fig.suptitle(f"Layerwise agreement across datasets ({mode}, metric={metric})", y=1.02, fontsize=18, fontweight="bold")
    fig.tight_layout()
    return fig, axes

In [ ]:
figB, axesB = plot_panel_B_layer_order_scatter(
    df_all=df_all_results,
    
    # dataset_a="tvsd",
    dataset_a="nsd_func1pt8mm_individualROIs",
    
    # dataset_b="things_fmri",
    dataset_b="tvsd",
    # dataset_b="bs_fz",
    
    metric="pearsonr_nc",   # or "pearsonr_nc"
    mode="rank",         # <-- this matches your “normalized layer order w.r.t. scores”
    superrois=("Early", "Mid", "High"),
    figsize=(18, 5),
    alpha=0.5,
    s=6,
)
plt.show()

plt.tight_layout()
figures_dir = save_dir
base_filename = 'layer_hierarchy_consistency_nsd_tvsd_rank'
formats = ['pdf', 'png', 'svg']
save_figs(figB, figures_dir, base_filename, formats=formats)

In [ ]:
figB, axesB = plot_panel_B_layer_order_scatter(
    df_all=df_all_results,
    
    # dataset_a="tvsd",
    dataset_a="nsd_func1pt8mm_individualROIs",
    
    # dataset_b="things_fmri",
    dataset_b="things_fmri",
    # dataset_b="bs_fz",
    
    metric="pearsonr_nc",   # or "pearsonr_nc"
    mode="rank",         # <-- this matches your “normalized layer order w.r.t. scores”
    superrois=("Early", "Mid", "High"),
    figsize=(18, 5),
    alpha=0.5,
    s=6,
)
plt.show()

plt.tight_layout()
figures_dir = save_dir
base_filename = 'layer_hierarchy_consistency_nsd_tfmri_rank'
formats = ['pdf', 'png', 'svg']
save_figs(figB, figures_dir, base_filename, formats=formats)

In [ ]:
# data = df_all_results[df_all_results["model_id"].apply(lambda x: 'deit' in x)].copy()
# data = df_all_results[df_all_results["model_id"].apply(lambda x: 'deit_' in x)].copy()
# # data = df_all_results[df_all_results["model_id"].apply(lambda x: 'convnext' in x)].copy()


data = df_all_results.copy()

In [ ]:
# Cell B3 (score-based version): same plot, but x/y are layerwise SCORES instead of rank_norm
# This uses the same helper functions you already have:
#   - compute_layerwise_rank_table(...)
#   - plot_panel_B_layer_order_scatter(...)

figB_score, axesB_score = plot_panel_B_layer_order_scatter(
    df_all=data,
    dataset_a="nsd_func1pt8mm_individualROIs",
    dataset_b="tvsd",
    metric="pearsonr_nc",   # or "pearsonr_nc" "cv_score"
    mode="score",        # <-- raw layerwise score scatter
    superrois=("Early", "Mid", "High"),
    figsize=(18, 5),
    alpha=0.5,
    s=6,
)

plt.show()

plt.tight_layout()
figures_dir = save_dir
base_filename = 'layer_hierarchy_consistency_nsd_tvsd_score'
formats = ['pdf', 'png', 'svg']
save_figs(figB_score, figures_dir, base_filename, formats=formats)

In [ ]:
# Cell B3 (score-based version): same plot, but x/y are layerwise SCORES instead of rank_norm
# This uses the same helper functions you already have:
#   - compute_layerwise_rank_table(...)
#   - plot_panel_B_layer_order_scatter(...)

figB_score, axesB_score = plot_panel_B_layer_order_scatter(
    df_all=data,
    dataset_a="nsd_func1pt8mm_individualROIs",
    dataset_b="things_fmri",
    metric="pearsonr_nc",   # or "pearsonr_nc" "cv_score"
    mode="score",        # <-- raw layerwise score scatter
    superrois=("Early", "Mid", "High"),
    figsize=(18, 5),
    alpha=0.5,
    s=6,
)

plt.show()

plt.tight_layout()
figures_dir = save_dir
base_filename = 'layer_hierarchy_consistency_nsd_tfmri_score'
formats = ['pdf', 'png', 'svg']
save_figs(figB_score, figures_dir, base_filename, formats=formats)